## WP002 — xG / Dixon-Coles Ablation

See `README.md` in this folder for methodology and results. Run the next cell to execute the ablation (resumable — safe to stop and re-run), then the cells after it load and compare all 4 arms (`both`, `neither`, `xg_only`, `dc_only`): MAE, pooled RPS, and paired bootstrap significance tests isolating each feature's individual contribution.

In [1]:
# Run the ablation: any arm without a complete checkpoint gets (re)run,
# window by window, each in its own subprocess — same isolation rationale
# as WP001 (see its README's "Why subprocess isolation" section): avoids
# accumulating JAX/compiled-program state across many window-fits.
# Resumable — re-running this cell picks up wherever each arm's checkpoint
# left off. To force a specific arm to redo from scratch (e.g. after a code
# fix), delete its cv_checkpoint_<arm>.pkl before running this cell.
import pickle
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path('/Users/hadiahmed/Documents/projects/football-predictor')
WP_DIR = REPO_ROOT / 'work_products' / 'wp002_xg_dc_ablation'
DATA_PATH = WP_DIR / 'cv_shared_data.pkl'
SCRIPT_PATH = REPO_ROOT / 'scripts' / 'run_cv_window.py'

WINDOW_TIMEOUT_SECONDS = 1200  # 20 min/window — a window that hangs even in
                                # its own fresh process gets killed and
                                # skipped rather than blocking the whole run.

ARMS = [
    ('both', True, True),
    ('neither', False, False),
    ('xg_only', True, False),
    ('dc_only', False, True),
]

with open(DATA_PATH, 'rb') as f:
    shared = pickle.load(f)
windows = shared['windows']
n_windows = len(windows)

def load_checkpoint(path):
    if path.exists():
        with open(path, 'rb') as f:
            return pickle.load(f)
    return {'results': [], 'cv_match_predictions': []}

for arm_name, use_xg, use_dc in ARMS:
    checkpoint_path = WP_DIR / f'cv_checkpoint_{arm_name}.pkl'
    completed = {r['window'] for r in load_checkpoint(checkpoint_path)['results']}
    print(f"\n{'#'*70}\nARM: {arm_name} (use_xg={use_xg}, use_dc={use_dc}) — "
          f"{len(completed)}/{n_windows} windows already done\n{'#'*70}")

    for i in range(1, n_windows + 1):
        if i in completed:
            continue
        print(f"\n{'='*60}\n[{arm_name}] WINDOW {i}/{n_windows}\n{'='*60}")
        try:
            subprocess.run(
                [sys.executable, str(SCRIPT_PATH),
                 '--data-path', str(DATA_PATH),
                 '--checkpoint-path', str(checkpoint_path),
                 '--window-index', str(i),
                 '--use-xg', str(use_xg),
                 '--use-dc', str(use_dc)],
                timeout=WINDOW_TIMEOUT_SECONDS,
                check=True,
            )
        except subprocess.TimeoutExpired:
            print(f"[{arm_name}] WINDOW {i} exceeded {WINDOW_TIMEOUT_SECONDS}s — "
                  "killed and skipped. Re-run this cell to retry it.")
        except subprocess.CalledProcessError:
            print(f"[{arm_name}] WINDOW {i} failed with a real error (see "
                  "traceback above) — skipped. Re-run this cell to retry it.")

print(f"\n{'#'*70}\nABLATION RUN COMPLETE\n{'#'*70}")
for arm_name, *_ in ARMS:
    checkpoint_path = WP_DIR / f'cv_checkpoint_{arm_name}.pkl'
    n_done = len(load_checkpoint(checkpoint_path)['results'])
    print(f"  {arm_name}: {n_done}/{n_windows} windows")


######################################################################
ARM: both (use_xg=True, use_dc=True) — 35/35 windows already done
######################################################################

######################################################################
ARM: neither (use_xg=False, use_dc=False) — 35/35 windows already done
######################################################################

######################################################################
ARM: xg_only (use_xg=True, use_dc=False) — 35/35 windows already done
######################################################################

######################################################################
ARM: dc_only (use_xg=False, use_dc=True) — 0/35 windows already done
######################################################################

[dc_only] WINDOW 1/35
[window 1/35] training rounds 1-36 (use_xg=False, use_dc=True)


Compiling.. :   0%|          | 0/4000 [00:00<?, ?it/s]

  0%|          | 0/4000 [00:00<?, ?it/s]

Running chain 1:   0%|          | 0/4000 [00:01<?, ?it/s]

Running chain 2:   0%|          | 0/4000 [00:01<?, ?it/s]

Running chain 1:   5%|▌         | 200/4000 [00:07<01:49, 34.61it/s]

Running chain 1:  10%|█         | 400/4000 [00:08<00:53, 67.56it/s]

Running chain 0:  20%|██        | 800/4000 [00:09<00:23, 138.61it/s][A

Running chain 0:  25%|██▌       | 1000/4000 [00:10<00:18, 159.77it/s][A

Running chain 0:  30%|███       | 1200/4000 [00:11<00:15, 178.73it/s]

Running chain 0:  35%|███▌      | 1400/4000 [00:11<00:13, 193.24it/s]

Running chain 0:  40%|████      | 1600/4000 [00:12<00:11, 208.48it/s]

Running chain 0:  45%|████▌     | 1800/4000 [00:13<00:10, 219.04it/s]

Running chain 0:  50%|█████     | 2000/4000 [00:14<00:09, 222.17it/s]

Running chain 0:  55%|█████▌    | 2200/4000 [00:15<00:07, 231.99it/s]

Running chain 0:  60%|██████    | 2400/4000 [00:16<00:06, 237.64it/s]

Runn

[window 1] MAE=1.145 LL_improvement=1.12
[window 1] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp002_xg_dc_ablation/cv_checkpoint_dc_only.pkl

[dc_only] WINDOW 2/35
[window 2/35] training rounds 1-41 (use_xg=False, use_dc=True)


Compiling.. :   0%|          | 0/4000 [00:00<?, ?it/s]

  0%|          | 0/4000 [00:00<?, ?it/s]

Running chain 1:   0%|          | 0/4000 [00:01<?, ?it/s]

Running chain 0:   5%|▌         | 200/4000 [00:06<01:42, 37.17it/s]

Running chain 0:  10%|█         | 400/4000 [00:07<00:51, 70.23it/s]

Running chain 0:  15%|█▌        | 600/4000 [00:09<00:34, 98.87it/s]

Running chain 0:  20%|██        | 800/4000 [00:10<00:26, 121.81it/s][A

Running chain 0:  25%|██▌       | 1000/4000 [00:11<00:21, 141.78it/s][A

Running chain 0:  30%|███       | 1200/4000 [00:12<00:18, 153.05it/s]

Running chain 0:  35%|███▌      | 1400/4000 [00:13<00:16, 162.23it/s]

Running chain 0:  40%|████      | 1600/4000 [00:14<00:14, 168.30it/s]

Running chain 0:  45%|████▌     | 1800/4000 [00:15<00:12, 180.72it/s]

Running chain 0:  50%|█████     | 2000/4000 [00:16<00:11, 181.45it/s]

Running chain 0:  55%|█████▌    | 2200/4000 [00:17<00:09, 194.18it/s]

Running chain 0:  60%|██████    | 2400/4000 [00:18<00:07, 203.97i

[window 2] MAE=1.291 LL_improvement=1.01
[window 2] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp002_xg_dc_ablation/cv_checkpoint_dc_only.pkl

[dc_only] WINDOW 3/35
[window 3/35] training rounds 1-46 (use_xg=False, use_dc=True)


...

[window 33] MAE=0.826 LL_improvement=-0.02
[window 33] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp002_xg_dc_ablation/cv_checkpoint_dc_only.pkl

[dc_only] WINDOW 34/35
[window 34/35] training rounds 1-201 (use_xg=False, use_dc=True)


Compiling.. :   0%|          | 0/4000 [00:00<?, ?it/s]

  0%|          | 0/4000 [00:00<?, ?it/s]

Running chain 1:   0%|          | 0/4000 [00:01<?, ?it/s]

Running chain 1:   5%|▌         | 200/4000 [00:43<13:21,  4.74it/s]

Running chain 1:  10%|█         | 400/4000 [00:53<06:58,  8.61it/s]

Running chain 1:  15%|█▌        | 600/4000 [01:02<04:44, 11.93it/s]

Running chain 1:  20%|██        | 800/4000 [01:11<03:39, 14.58it/s]

Running chain 0:  25%|██▌       | 1000/4000 [01:17<03:00, 16.63it/s][A

Running chain 0:  30%|███       | 1200/4000 [01:25<02:32, 18.34it/s]

Running chain 1:  35%|███▌      | 1400/4000 [01:39<02:14, 19.36it/s]

Running chain 1:  40%|████      | 1600/4000 [01:47<01:58, 20.33it/s]

Running chain 1:  45%|████▌     | 1800/4000 [01:56<01:43, 21.17it/s]

Running chain 0:  50%|█████     | 2000/4000 [02:00<01:32, 21.65it/s]

Running chain 0:  55%|█████▌    | 2200/4000 [02:09<01:22, 21.93it/s]

Running chain 0:  60%|██████    | 2400/4000 [02:18<01:12, 22.14it/s]

Runni

[window 34] MAE=0.874 LL_improvement=0.69
[window 34] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp002_xg_dc_ablation/cv_checkpoint_dc_only.pkl

[dc_only] WINDOW 35/35
[window 35/35] training rounds 1-206 (use_xg=False, use_dc=True)


Compiling.. :   0%|          | 0/4000 [00:00<?, ?it/s]

  0%|          | 0/4000 [00:00<?, ?it/s]

Running chain 1:   0%|          | 0/4000 [00:01<?, ?it/s]

Running chain 0:   5%|▌         | 200/4000 [00:42<13:03,  4.85it/s]

Running chain 1:   5%|▌         | 200/4000 [00:43<13:21,  4.74it/s]

Running chain 1:  10%|█         | 400/4000 [00:55<07:19,  8.20it/s]

Running chain 1:  15%|█▌        | 600/4000 [01:06<05:10, 10.95it/s]

Running chain 1:  20%|██        | 800/4000 [01:16<03:57, 13.47it/s]

Running chain 1:  25%|██▌       | 1000/4000 [01:25<03:14, 15.42it/s]

Running chain 1:  30%|███       | 1200/4000 [01:35<02:46, 16.82it/s]

Running chain 1:  35%|███▌      | 1400/4000 [01:45<02:23, 18.08it/s]

Running chain 1:  40%|████      | 1600/4000 [01:54<02:06, 19.03it/s]

Running chain 1:  45%|████▌     | 1800/4000 [02:03<01:52, 19.61it/s]

Running chain 1:  50%|█████     | 2000/4000 [02:13<01:40, 19.80it/s]

Running chain 1:  55%|█████▌    | 2200/4000 [02:23<01:28, 20.31it/s]

Running 

[window 35] MAE=0.916 LL_improvement=0.70
[window 35] checkpoint updated: /Users/hadiahmed/Documents/projects/football-predictor/work_products/wp002_xg_dc_ablation/cv_checkpoint_dc_only.pkl

######################################################################
ABLATION RUN COMPLETE
######################################################################
  both: 35/35 windows
  neither: 35/35 windows
  xg_only: 35/35 windows
  dc_only: 35/35 windows


In [2]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import poisson

WP_DIR = Path('/Users/hadiahmed/Documents/projects/football-predictor/work_products/wp002_xg_dc_ablation')

with open(WP_DIR / 'cv_shared_data.pkl', 'rb') as f:
    shared = pickle.load(f)
df_cv = shared['df_cv']
windows = shared['windows']

ARM_NAMES = ['both', 'neither', 'xg_only', 'dc_only']
arms = {}
for name in ARM_NAMES:
    path = WP_DIR / f'cv_checkpoint_{name}.pkl'
    if path.exists():
        with open(path, 'rb') as f:
            arms[name] = pickle.load(f)
    else:
        arms[name] = None

for name in ARM_NAMES:
    if arms[name] is None:
        print(f"{name}: NOT FOUND (run the cell above first)")
    else:
        print(f"{name}: {len(arms[name]['results'])}/{len(windows)} windows, "
              f"{len(arms[name]['cv_match_predictions'])} match predictions")

both: 35/35 windows, 401 match predictions
neither: 35/35 windows, 401 match predictions
xg_only: 35/35 windows, 401 match predictions
dc_only: 35/35 windows, 401 match predictions


In [3]:
# Evaluation utilities (same as WP001) — see WP001's README for what each means.
#
# outcome_probs_from_lambda delegates to football_model.model.predict's
# dc_outcome_probs instead of re-deriving the scoreline grid here — that's
# the one place the Dixon-Coles correction is implemented, so dc_only/both
# actually get scored with it applied instead of silently as if use_dc=False
# (a real gap found after this notebook first ran — see README).
from football_model.model.predict import dc_outcome_probs as _dc_outcome_probs


def outcome_probs_from_lambda(lambda_home, lambda_away, rho_dc=None, max_goals=10):
    return _dc_outcome_probs(lambda_home, lambda_away, rho=rho_dc, max_goals=max_goals)


def match_outcome(goals_home, goals_away):
    if goals_home > goals_away:
        return 'H'
    if goals_home == goals_away:
        return 'D'
    return 'A'


def rps_home_draw_away(p_home, p_draw, p_away, actual):
    cp1, cp2 = p_home, p_home + p_draw
    ce1 = 1.0 if actual == 'H' else 0.0
    ce2 = 1.0 if actual in ('H', 'D') else 0.0
    return 0.5 * ((cp1 - ce1) ** 2 + (cp2 - ce2) ** 2)


def reliability_table(pred_probs, actual_flags, n_bins=5):
    pred_probs = np.asarray(pred_probs)
    actual_flags = np.asarray(actual_flags)
    bins = np.linspace(0, 1, n_bins + 1)
    bin_idx = np.clip(np.digitize(pred_probs, bins[1:-1]), 0, n_bins - 1)
    rows = []
    for b in range(n_bins):
        mask = bin_idx == b
        n = int(mask.sum())
        if n == 0:
            continue
        rows.append({'bin': f"{bins[b]:.1f}-{bins[b+1]:.1f}", 'n_matches': n,
                      'mean_predicted': pred_probs[mask].mean(), 'actual_frequency': actual_flags[mask].mean()})
    return pd.DataFrame(rows)


def bootstrap_mean_ci(values, n_boot=5000, alpha=0.05, seed=0):
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    n = len(values)
    boot_means = np.array([rng.choice(values, size=n, replace=True).mean() for _ in range(n_boot)])
    lo, hi = np.percentile(boot_means, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return values.mean(), lo, hi

In [4]:
# Per-arm summary: MAE and mean LL improvement over naive (with bootstrap CI)
summary_rows = []
for name in ARM_NAMES:
    if arms[name] is None:
        continue
    results_df = pd.DataFrame(arms[name]['results'])
    mean_imp, lo, hi = bootstrap_mean_ci(results_df['ll_improvement'].values)
    summary_rows.append({
        'arm': name,
        'n_windows': len(results_df),
        'mae': results_df['mae'].mean(),
        'll_improvement_mean': mean_imp,
        'll_improvement_ci': f"[{lo:.2f}, {hi:.2f}]",
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

    arm  n_windows      mae  ll_improvement_mean ll_improvement_ci
   both         35 0.921169             1.597714      [0.99, 2.23]
neither         35 0.920986             1.533543      [0.93, 2.15]
xg_only         35 0.921437             1.568308      [0.97, 2.19]
dc_only         35 0.920958             1.562257      [0.95, 2.20]


In [5]:
# Pooled RPS per arm — the metric we actually trust (see WP001 README)
historical_avg_home = df_cv['goals_home'].mean()
historical_avg_away = df_cv['goals_away'].mean()

arm_match_dfs = {}
rps_rows = []
for name in ARM_NAMES:
    if arms[name] is None:
        continue
    mdf = pd.DataFrame(arms[name]['cv_match_predictions']).sort_values('window').reset_index(drop=True)
    mdf['idx_in_window'] = mdf.groupby('window').cumcount()
    mdf['outcome'] = [match_outcome(r.goals_home, r.goals_away) for r in mdf.itertuples()]
    # rho_dc: None for neither/xg_only (use_dc=False, so outcome_probs_from_lambda
    # is plain independent Poisson); each match's trained rho_dc for dc_only/both.
    mdf['probs'] = [
        outcome_probs_from_lambda(r.lambda_home, r.lambda_away, rho_dc=getattr(r, 'rho_dc', None))
        for r in mdf.itertuples()
    ]
    mdf['rps'] = [rps_home_draw_away(*p, a) for p, a in zip(mdf['probs'], mdf['outcome'])]
    arm_match_dfs[name] = mdf
    rps_rows.append({'arm': name, 'n_matches': len(mdf), 'pooled_rps': mdf['rps'].mean()})

naive_probs = outcome_probs_from_lambda(historical_avg_home, historical_avg_away)
print(f"Naive baseline RPS reference (same for all arms — league-average goal rate)")

rps_df = pd.DataFrame(rps_rows)
print(rps_df.to_string(index=False))

Naive baseline RPS reference (same for all arms — league-average goal rate)
    arm  n_matches  pooled_rps
   both        401    0.196285
neither        401    0.196393
xg_only        401    0.196362
dc_only        401    0.196329


In [6]:
# Paired comparisons — all 4 arms are scored on the IDENTICAL windows/test
# matches, so we can pair up predictions match-for-match instead of
# comparing unpaired distributions. This is a much more powerful test for
# isolating one feature's marginal effect than comparing each arm to naive
# separately, since it cancels out match-to-match difficulty/noise that
# affects every arm equally.

def paired_rps_diff(arm_a, arm_b):
    """RPS(arm_b) - RPS(arm_a), matched by (window, idx_in_window).
    Positive means arm_a has LOWER (better) RPS than arm_b."""
    a = arm_match_dfs[arm_a].set_index(['window', 'idx_in_window'])
    b = arm_match_dfs[arm_b].set_index(['window', 'idx_in_window'])
    joined = a[['rps', 'goals_home', 'goals_away']].join(
        b[['rps', 'goals_home', 'goals_away']], lsuffix='_a', rsuffix='_b', how='inner'
    )
    # sanity check: same match should have the same actual result in both arms
    assert (joined['goals_home_a'] == joined['goals_home_b']).all()
    assert (joined['goals_away_a'] == joined['goals_away_b']).all()
    return (joined['rps_b'] - joined['rps_a']).values, len(joined)

comparisons = [
    ('both', 'neither', "Both features vs neither (total combined effect)"),
    ('both', 'dc_only', "xG's marginal contribution on top of Dixon-Coles"),
    ('both', 'xg_only', "Dixon-Coles' marginal contribution on top of xG"),
    ('xg_only', 'neither', "xG alone vs neither"),
    ('dc_only', 'neither', "Dixon-Coles alone vs neither"),
]

comparison_rows = []
for better_arm, worse_arm, label in comparisons:
    if arms[better_arm] is None or arms[worse_arm] is None:
        continue
    diffs, n = paired_rps_diff(better_arm, worse_arm)
    mean_diff, lo, hi = bootstrap_mean_ci(diffs)
    comparison_rows.append({
        'comparison': label,
        'n_matches': n,
        'mean_rps_improvement': mean_diff,
        '95%_ci': f"[{lo:.4f}, {hi:.4f}]",
        'significant': 'YES' if lo > 0 else ('NEGATIVE' if hi < 0 else 'no'),
    })

comparison_df = pd.DataFrame(comparison_rows)
print(comparison_df.to_string(index=False))

                                      comparison  n_matches  mean_rps_improvement            95%_ci significant
Both features vs neither (total combined effect)        401              0.000108 [-0.0005, 0.0008]          no
xG's marginal contribution on top of Dixon-Coles        401              0.000044 [-0.0004, 0.0005]          no
 Dixon-Coles' marginal contribution on top of xG        401              0.000077 [-0.0004, 0.0005]          no
                             xG alone vs neither        401              0.000031 [-0.0004, 0.0005]          no
                    Dixon-Coles alone vs neither        401              0.000064 [-0.0004, 0.0005]          no
